# 03 – Data Preprocessing

**Phase 2: Preprocessing** · Group AI-39

This notebook prepares the football match dataset for the match outcome prediction task. It creates the **Home Win**, **Draw**, and **Away Win** target, orders fixtures chronologically, removes leakage-prone or inappropriate columns, and demonstrates exclusion of the 2008/09 warm-up season.

The prepared data is intended for historical feature engineering. The main `processed` dataframe retains all seasons so the next stage can calculate cross-season history before excluding the warm-up season. Reusable preprocessing logic is implemented in `src/preprocessing.py`.

## 1. Imports

All transformations below use the project's shared database loader and preprocessing functions.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.db import load_matches
from src.preprocessing import (
    add_match_outcome,
    sort_matches_chronologically,
    remove_excluded_columns,
    remove_warmup_season,
)

## 2. Load match data

Load the match table using `src.db.load_matches()`. The loader parses match dates and provides the project-standard input.

In [2]:
matches = load_matches()

print(f"Shape: {matches.shape}")
print(f"Column count: {len(matches.columns)}")
display(matches.head())
display(pd.Series(matches.columns, name="input columns"))

Shape: (25979, 119)
Column count: 119


,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,GBH,GBD,GBA,BSH,BSD,BSA,country_name,league_name,home_team_name,away_team_name
0,1,1,1,2008/2009,1,2008-08-17,492473,9987,9993,1,...,1.78,3.25,4.00,1.73,3.40,4.20,Belgium,Belgium Jupiler League,KRC Genk,Beerschot AC
1,2,1,1,2008/2009,1,2008-08-16,492474,10000,9994,0,...,1.85,3.25,3.75,1.91,3.25,3.60,Belgium,Belgium Jupiler League,SV Zulte-Waregem,Sporting Lokeren
2,3,1,1,2008/2009,1,2008-08-16,492475,9984,8635,0,...,2.50,3.20,2.50,2.30,3.20,2.75,Belgium,Belgium Jupiler League,KSV Cercle Brugge,RSC Anderlecht
3,4,1,1,2008/2009,1,2008-08-17,492476,9991,9998,5,...,1.50,3.75,5.50,1.44,3.75,6.50,Belgium,Belgium Jupiler League,KAA Gent,RAEC Mons
4,5,1,1,2008/2009,1,2008-08-16,492477,7947,9985,1,...,4.50,3.50,1.65,4.75,3.30,1.67,Belgium,Belgium Jupiler League,FCV Dender EH,Standard de Liège


0                  id
1          country_id
2           league_id
3              season
4               stage
            ...      
114               BSA
115      country_name
116       league_name
117    home_team_name
118    away_team_name
Name: input columns, Length: 119, dtype: str

## 3. Initial data check

Review dataset size, missingness, match-ID uniqueness, and the observed date range before transforming the data.

In [3]:
initial_check = pd.Series({
    "rows": len(matches),
    "columns": len(matches.columns),
    "missing cells": int(matches.isna().sum().sum()),
    "duplicate match IDs": int(matches["id"].duplicated().sum()),
    "first match date": matches["date"].min(),
    "last match date": matches["date"].max(),
}, name="value")
display(initial_check)

initial_missing = matches.isna().sum().sort_values(ascending=False)
display(initial_missing[initial_missing > 0].rename("missing values").to_frame())

rows                                 25979
columns                                119
missing cells                       407395
duplicate match IDs                      0
first match date       2008-07-18 00:00:00
last match date        2016-05-25 00:00:00
Name: value, dtype: object

,missing values
PSH,14811
PSD,14811
PSA,14811
BSD,11818
BSH,11818
...,...
home_player_9,1273
away_player_7,1235
away_player_1,1234
home_player_7,1227


## 4. Create the match outcome target

The target is derived only from `home_team_goal` and `away_team_goal`:

- **Home Win** when home goals are greater than away goals.
- **Draw** when home goals equal away goals.
- **Away Win** when home goals are fewer than away goals.

In [4]:
processed = add_match_outcome(matches)

outcome_counts = processed["match_outcome"].value_counts()
display(outcome_counts.rename("matches").to_frame())
display(processed["match_outcome"].value_counts(normalize=True).rename("proportion").to_frame())
unknown_targets = int(processed["match_outcome"].eq("Unknown").sum())
print(f"Unknown target values: {unknown_targets}")
assert unknown_targets == 0

,matches
match_outcome,
Home Win,11917
Away Win,7466
Draw,6596


,proportion
match_outcome,
Home Win,0.458717
Away Win,0.287386
Draw,0.253897


Unknown target values: 0


## 5. Chronological sorting

Fixtures are ordered by date and match ID. Chronological ordering is required because later historical feature engineering must use only information available before each match.

In [5]:
processed = sort_matches_chronologically(processed)

date_ordered = processed["date"].is_monotonic_increasing
date_id_ordered = processed[["date", "id"]].equals(
    processed.sort_values(["date", "id"])[["date", "id"]].reset_index(drop=True)
)
print(f"Dates are chronological: {date_ordered}")
print(f"Rows follow date then match-ID order: {date_id_ordered}")
display(processed[["id", "date", "season"]].head())
display(processed[["id", "date", "season"]].tail())
assert date_ordered and date_id_ordered

Dates are chronological: True
Rows follow date then match-ID order: True


,id,date,season
0,24559,2008-07-18,2008/2009
1,24560,2008-07-19,2008/2009
2,24561,2008-07-20,2008/2009
3,24562,2008-07-20,2008/2009
4,24613,2008-07-23,2008/2009


,id,date,season
25974,25945,2016-05-25,2015/2016
25975,25946,2016-05-25,2015/2016
25976,25947,2016-05-25,2015/2016
25977,25948,2016-05-25,2015/2016
25978,25949,2016-05-25,2015/2016


## 6. Remove excluded columns

Use `remove_excluded_columns()` to remove in-match event/XML data, player lineup and X/Y-position data, and betting odds. The removed-column lists and category counts below are calculated from the actual input and output columns.

In [6]:
before_columns = set(processed.columns)
processed = remove_excluded_columns(processed)
removed_columns = sorted(before_columns - set(processed.columns))

lineup_columns_removed = [
    column for column in removed_columns
    if column.startswith(("home_player_X", "away_player_X", "home_player_Y", "away_player_Y", "home_player_", "away_player_"))
]
odds_columns_removed = [
    column for column in removed_columns
    if column[:1].isupper() and column[-1:] in {"H", "D", "A"}
]
event_columns_removed = [
    column for column in removed_columns
    if column not in lineup_columns_removed and column not in odds_columns_removed
]

print(f"Total columns removed: {len(removed_columns)}")
display(pd.Series({
    "in-match event/XML columns": len(event_columns_removed),
    "player lineup/X-Y columns": len(lineup_columns_removed),
    "betting odds columns": len(odds_columns_removed),
}, name="removed count").to_frame())
display(pd.Series(removed_columns, name="removed column"))

Total columns removed: 104


,removed count
in-match event/XML columns,8
player lineup/X-Y columns,66
betting odds columns,30


0               B365A
1               B365D
2               B365H
3                 BSA
4                 BSD
            ...      
99     home_player_Y8
100    home_player_Y9
101        possession
102           shotoff
103            shoton
Name: removed column, Length: 104, dtype: str

## 7. Verify target and core columns

The match identifiers, fixture context, full-time goals (for the target definition), and target remain available at this preprocessing stage. The goal columns are outcome inputs here, not predictor features in the final modelling feature set.

In [7]:
core_columns = [
    "id", "date", "season", "league_id",
    "home_team_api_id", "away_team_api_id",
    "home_team_goal", "away_team_goal", "match_outcome",
]
display(processed[core_columns].head())
print(f"All core columns present: {set(core_columns).issubset(processed.columns)}")
print(f"Target classes valid: {processed['match_outcome'].isin(['Home Win', 'Draw', 'Away Win']).all()}")
print(f"Match IDs unique: {processed['id'].is_unique}")
assert set(core_columns).issubset(processed.columns)
assert processed["match_outcome"].isin(["Home Win", "Draw", "Away Win"]).all()
assert processed["id"].is_unique

,id,date,season,league_id,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,match_outcome
0,24559,2008-07-18,2008/2009,24558,10192,9931,1,2,Away Win
1,24560,2008-07-19,2008/2009,24558,9930,10179,3,1,Home Win
2,24561,2008-07-20,2008/2009,24558,10199,9824,1,2,Away Win
3,24562,2008-07-20,2008/2009,24558,7955,10243,1,2,Away Win
4,24613,2008-07-23,2008/2009,24558,9931,9956,1,0,Home Win


All core columns present: True
Target classes valid: True
Match IDs unique: True


## 8. Demonstrate warm-up season exclusion

The 2008/09 season is treated as a warm-up because teams have limited prior-match history for historical feature generation. The project feature-engineering pipeline calculates historical features across **all seasons first**, then removes 2008/09. Accordingly, this cell applies `remove_warmup_season()` to a separate copy: `processed` retains all seasons as the input needed for that next stage.

In [8]:
before_rows = len(processed)
processed_without_warmup = remove_warmup_season(processed)
after_rows = len(processed_without_warmup)

print(f"Rows before warm-up exclusion: {before_rows:,}")
print(f"Rows after warm-up exclusion: {after_rows:,}")
print(f"Rows removed: {before_rows - after_rows:,}")
print("Seasons retained after warm-up exclusion:")
display(processed_without_warmup["season"].value_counts().sort_index().rename("matches").to_frame())
print(f"Main processed dataframe still contains all seasons: {processed['season'].nunique() == matches['season'].nunique()}")

Rows before warm-up exclusion: 25,979
Rows after warm-up exclusion: 22,653
Rows removed: 3,326
Seasons retained after warm-up exclusion:


,matches
season,
2009/2010,3230
2010/2011,3260
2011/2012,3220
2012/2013,3260
2013/2014,3032
2014/2015,3325
2015/2016,3326


Main processed dataframe still contains all seasons: True


## 9. Final preprocessing validation

Validate the all-season preprocessed dataframe that is handed to historical feature engineering. The warm-up-excluded copy above is for inspection; the downstream feature pipeline excludes that season only after calculating cross-season historical features.

In [9]:
assert processed["match_outcome"].isin(["Home Win", "Draw", "Away Win"]).all()
assert processed["id"].is_unique
assert processed["date"].is_monotonic_increasing

final_check = pd.Series({
    "rows": len(processed),
    "columns": len(processed.columns),
    "missing cells": int(processed.isna().sum().sum()),
    "duplicate match IDs": int(processed["id"].duplicated().sum()),
    "dates chronological": processed["date"].is_monotonic_increasing,
}, name="value")
display(final_check)
display(processed["match_outcome"].value_counts().rename("matches").to_frame())
display(processed["season"].value_counts().sort_index().rename("matches").to_frame())
display(processed.isna().sum().sort_values(ascending=False).rename("missing values").to_frame().head(15))

rows                   25979
columns                   16
missing cells              0
duplicate match IDs        0
dates chronological     True
Name: value, dtype: object

,matches
match_outcome,
Home Win,11917
Away Win,7466
Draw,6596


,matches
season,
2008/2009,3326
2009/2010,3230
2010/2011,3260
2011/2012,3220
2012/2013,3260
2013/2014,3032
2014/2015,3325
2015/2016,3326


,missing values
id,0
country_id,0
league_id,0
season,0
stage,0
date,0
match_api_id,0
home_team_api_id,0
away_team_api_id,0
home_team_goal,0


## 10. Final preprocessing summary

- The match outcome target was created from full-time home and away goals.
- Matches were sorted chronologically by date and match ID.
- In-match event/XML, lineup/player X-Y, and betting-odds columns were removed using the shared preprocessing function.
- The 2008/09 warm-up exclusion was demonstrated using a separate dataframe; the all-season `processed` data is retained so historical features can use cross-season history.
- The preprocessed data passed target, unique-ID, and chronological-order checks and is ready for historical feature engineering.

## 11. Important handoff note

The next stage is historical feature engineering in `notebooks/04_feature_engineering.ipynb`. Historical features must use strictly earlier matches only. Leakage prevention is handled by shifted historical calculations (`shift(1)`) in `src/feature_engineering.py`; the 2008/09 warm-up season is removed after those calculations.